In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer,
    pipeline
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import evaluate
from tqdm.auto import tqdm

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
set_seed(42)

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load emotion dataset
dataset = load_dataset("emotion")

# Display dataset structure
print(dataset)

# Show class labels
label_names = dataset["train"].features["label"].names
print(f"Class labels: {label_names}")
print(f"Number of classes: {len(label_names)}")

# Show dataset sizes
print(f"Train size: {len(dataset['train'])}")
print(f"Validation size: {len(dataset['validation'])}")
print(f"Test size: {len(dataset['test'])}")

# Display 5 random examples
for i in range(5):
    sample = dataset["train"][i]
    print(f"Text: {sample['text']}")
    print(f"Label: {sample['label']} ({label_names[sample['label']]})")
    print("-" * 50)

In [ ]:
# Load tokenizer
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Tokenize a few examples manually to understand the process
texts = [
    "i feel very happy today",
    "i am so sad about this news",
    "this is a relatively longer text that might need some truncation because it exceeds the maximum length limit"
]

for text in texts:
    # Tokenize with truncation and padding
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text, truncation=True, padding="max_length", max_length=10)
    
    print(f"\nOriginal text: {text}")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {token_ids}")
    print(f"Special tokens: [CLS] (id: {tokenizer.cls_token_id}), [SEP] (id: {tokenizer.sep_token_id}), [PAD] (id: {tokenizer.pad_token_id})")
    print(f"Attention mask: {[1 if id != tokenizer.pad_token_id else 0 for id in token_ids]}")

# Tokenize the entire dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
# Create pipeline for text classification
pretrained_pipeline = pipeline(
    "text-classification", 
    model=model_checkpoint,
    device=0 if torch.cuda.is_available() else -1
)

# Test on a few examples
test_texts = [
    "i am so excited about the party tonight",
    "i feel really down and lonely",
    "this movie made me cry",
    "i love you so much",
    "i'm afraid of the dark"
]

print("Pre-trained model predictions (before fine-tuning):")
for text in test_texts:
    result = pretrained_pipeline(text)
    print(f"Text: {text}")
    print(f"Prediction: {result[0]['label']}, Confidence: {result[0]['score']:.4f}")
    print("-" * 50)

In [ ]:
# Load model for sequence classification with correct number of labels
num_labels = len(label_names)
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=num_labels
)
model.to(device)

# Remove 'text' column and rename 'label' to 'labels' for Trainer compatibility
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

# Define evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")
    return {"accuracy": accuracy, "f1_macro": f1_macro}

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    seed=42,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# Train the model
trainer.train()

# Evaluate on test set
test_results = trainer.evaluate(tokenized_datasets["test"])
print(f"Test Results: {test_results}")

# Get predictions on test set for confusion matrix
predictions = trainer.predict(tokenized_datasets["test"])
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

In [ ]:
# Create sample_predictions.csv
sample_indices = list(range(min(50, len(true_labels))))
sample_texts = dataset["test"]["text"][:len(sample_indices)]

predictions_df = pd.DataFrame({
    "text": sample_texts,
    "true_label": [label_names[label] for label in true_labels[sample_indices]],
    "pred_label": [label_names[label] for label in pred_labels[sample_indices]],
    "confidence": [max(row) for row in predictions.predictions[sample_indices]]
})

predictions_df.to_csv("./artifacts/sample_predictions.csv", index=False)
print("Saved sample_predictions.csv")

# Create confusion matrix
cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig("./artifacts/confusion_matrix.png", dpi=300)
print("Saved confusion_matrix.png")

# Optional: Print classification report
print("\nClassification Report:")
print(classification_report(true_labels, pred_labels, target_names=label_names))